# Week 23 — Safe exploration, LTL shielding, CBFs, robust RL

This workbook has two explicit exercise modes, controlled by `REFERENCE_MODE`
in the setup cell.

- **Reference (`True`, default):** supplied implementations demonstrate the
  calculations. Passing these checks verifies the reference, not your code.
- **Learner (`False`):** complete each scaffold's function, then restart and
  run all cells. The corresponding checks call your function. An unfinished
  scaffold raises `NotImplementedError`; there is no reference fallback.
  In a reference-wrapper scaffold, replace its guarded placeholder body with
  your implementation before choosing learner mode.

Change the setup flag before restarting. Record a boundary-case prediction
before each check and change an input after it passes. Later experiment cells
without a scaffold are reference demonstrations, not automated learner grades.

**Teaching companion:** [Worked examples, assumptions and self-checks](teaching_guide.md).
For additional independently graded tasks, see the
[progressive coding projects](../docs/INTERACTIVE_LEARNING.md).


**A bounded first investigation:** Test an empty safe set before a long rollout. Start with **Exercises 2, 3 and 7** and the
[guide’s prediction, visual diagnostic and changed-input check](teaching_guide.md#notebook-investigation-test-an-empty-safe-set-before-a-long-rollout).
Save your own prediction before running the reference cells. For your own implementation, choose learner mode in setup, restart, and verify
that the corresponding check calls your completed function.


**Worked examples and derivations:** [Teaching guide](teaching_guide.md) — prerequisite links, analogy boundaries, calculated examples, code connections, and retrieval questions.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))
from common.plotting import use_rl_style, RL_PALETTE
from common.seed import seed_everything
use_rl_style(); seed_everything(0)
import numpy as np, matplotlib.pyplot as plt
from project import solutions as ref
from project.env_grid import DangerGridWorld
env = DangerGridWorld(); env.reset(seed=0)
print(env.render_ascii())
# True runs documented references. False checks YOUR completed functions.
# Restart and run all after changing this flag; an unfinished learner stops.
REFERENCE_MODE = True
def select_exercise(reference, learner):
    return reference if REFERENCE_MODE else learner
print("Exercise mode:", "reference demonstration" if REFERENCE_MODE else "learner checks (no fallback)")


## Exercise 1 ★ — DFA for G(¬danger)

Build the 2-state safety automaton.

> **Hint**: states `q0` (accepting) and `q_bad` (rejecting sink); `danger` sends `q0` to `q_bad` and absorbs `q_bad`.

In [ ]:
def exercise_1():
    # YOUR CODE HERE
    if not REFERENCE_MODE:
        raise NotImplementedError("Implement exercise_1 before running learner checks")
    return ref.exercise_1()


In [ ]:
dfa = exercise_1()
assert dfa['initial'] == 'q0' and 'q_bad' not in dfa['accepting']
print(dfa)

## Exercise 2 ★ — One-step unsafe predicate

Return True iff `action a` in `state s` would enter a danger cell.

> **Hint**: dynamics are deterministic; use `env.adjacent_dangerous`.

In [ ]:
def exercise_2(env, s_idx, a):
    # YOUR CODE HERE
    if not REFERENCE_MODE:
        raise NotImplementedError("Implement exercise_2 before running learner checks")
    return ref.exercise_2(env, s_idx, a)


In [ ]:
assert exercise_2(env, env.s2idx((1,1)), 1) == True
assert exercise_2(env, env.s2idx((0,0)), 2) == False
print('ok')

## Exercise 3 ★★ — Pre-shield filtering

Given an action preference list, return the first SAFE action.

> **Hint**: iterate through preferences; return first one whose lookahead is safe.

In [ ]:
def exercise_3(env, preferences, s_idx):
    # YOUR CODE HERE
    if not REFERENCE_MODE:
        raise NotImplementedError("Implement exercise_3 before running learner checks")
    return ref.exercise_3(env, preferences, s_idx)


In [ ]:
out = exercise_3(env, [1, 2, 0, 3], env.s2idx((1,1)))
assert out == 0  # first safe action in the stated preference order
print('chose', out)

## Exercise 4 ★★ — Shielded vs unshielded Q-learning

Train both, compare cumulative unsafe transitions.

In [ ]:
def exercise_4(seed=0, n_episodes=200):
    # YOUR CODE HERE
    if not REFERENCE_MODE:
        raise NotImplementedError("Implement exercise_4 before running learner checks")
    return ref.exercise_4(seed=seed, n_episodes=n_episodes)


In [ ]:
res = exercise_4(seed=0, n_episodes=200)
print('shielded unsafe', res['shielded']['total_unsafe'])
print('unshielded unsafe', res['unshielded']['total_unsafe'])
fig, ax = plt.subplots(figsize=(7,3.5))
ax.plot(np.cumsum(res['unshielded']['unsafe']), color=RL_PALETTE['red'], label='unshielded')
ax.plot(np.cumsum(res['shielded']['unsafe']), color=RL_PALETTE['teal'], label='shielded')
ax.set_xlabel('Episode'); ax.set_ylabel('Cumulative unsafe transitions'); ax.legend(); plt.show()

## Exercise 5 ★★ — Check a finite rollout against the stated safety property

Count unsafe transitions on the supplied deterministic grid. Separately state the assumptions for soundness: safe initialization, a correct transition model, feasible safe actions, and exact execution. Zero observed violations is a finite check, not a proof of the theorem.

In [ ]:
def exercise_5(seed=0, n_episodes=300):
    if not REFERENCE_MODE:
        raise NotImplementedError("Implement exercise_5 before running learner checks")
    return ref.exercise_5(seed=seed, n_episodes=n_episodes)


In [ ]:
n = exercise_5(seed=0, n_episodes=300)
assert n == 0, f'shield was violated {n} times'
print('ok, zero violations under shield')

## Exercise 6 ★★ — Meaningful objective: DR vs fixed-length pendulum

Train a tiny policy gradient agent both with and without domain randomisation, evaluate at OOD lengths.

> **Hint**: this is the matching/safety hook — domain randomisation changes the training distribution. Compare its average and worst measured evaluation return; success under a new shift is an empirical question.

In [ ]:
def exercise_6(seed=0, n_epochs=15, steps_per_epoch=300):
    if not REFERENCE_MODE:
        raise NotImplementedError("Implement exercise_6 before running learner checks")
    return ref.exercise_6(seed=seed, n_epochs=n_epochs, steps_per_epoch=steps_per_epoch)


In [ ]:
res = exercise_6(seed=0, n_epochs=15, steps_per_epoch=300)
print('DR eval:   ', res['dr']['eval'])
print('FIXED eval:', res['fixed']['eval'])
Ls = sorted(res['dr']['eval'].keys())
fig, ax = plt.subplots(figsize=(7,3.5))
ax.plot(Ls, [res['dr']['eval'][L] for L in Ls], 'o-', label='DR', color=RL_PALETTE['blue'])
ax.plot(Ls, [res['fixed']['eval'][L] for L in Ls], 's--', label='fixed', color=RL_PALETTE['orange'])
ax.axvspan(0.5,1.5, color=RL_PALETTE['teal'], alpha=0.1, label='DR train support')
ax.set_xlabel('Length'); ax.set_ylabel('Eval return'); ax.legend(); plt.show()

## Exercise 7 ★★★ — CBF projection in 1D closed form

For scalar `a`, scalar `h` and `dh/da`, project `a_proposed` onto the half-space `dh/da · a >= -alpha · h`.

> **Hint**: only project when the constraint is violated; otherwise pass through.

In [ ]:
def exercise_7(a_proposed, h_val, dh_da, alpha=1.0):
    # YOUR CODE HERE
    if not REFERENCE_MODE:
        raise NotImplementedError("Implement exercise_7 before running learner checks")
    return ref.exercise_7(a_proposed, h_val, dh_da, alpha)


In [ ]:
assert abs(exercise_7(0.5, 2.0, 1.0, 1.0) - 0.5) < 1e-9
assert abs(exercise_7(2.0, 1.0, -1.0, 1.0) - 1.0) < 1e-9
print('ok')

## Exercise 8 ★★★ — Adversarial OOD evaluation

Train a DR policy on lengths [0.5, 1.5] then evaluate at extreme lengths (0.25, 2.0). Discuss in markdown how rapidly the policy degrades.

> **Hint**: plot return against evaluation length and mark the training interval. There is no universal distance or multiplier beyond which a policy must fail; report this model’s observed failure cases.

In [ ]:
def exercise_8(seed=0, n_epochs=15, steps_per_epoch=300):
    if not REFERENCE_MODE:
        raise NotImplementedError("Implement exercise_8 before running learner checks")
    return ref.exercise_8(seed=seed, n_epochs=n_epochs, steps_per_epoch=steps_per_epoch)


In [ ]:
ev = exercise_8(seed=0)
print(ev)

<!-- AUTO-HINTS-SECTION -->
## Hints toward answers

Attempt the exercise first. Before opening a hint, write the quantity you are
estimating, the data source, and the expected update direction. The
[teaching guide](teaching_guide.md) supplies worked examples, assumptions,
and transfer questions. The hints below collect the exercise's own inline
hints; they do not reveal or execute reference solutions.

<details>
<summary>Exercise 1 ★ — DFA for G(¬danger)</summary>

> **Hint**: states `q0` (accepting) and `q_bad` (rejecting sink); `danger` sends `q0` to `q_bad` and absorbs `q_bad`.

</details>

<details>
<summary>Exercise 2 ★ — One-step unsafe predicate</summary>

> **Hint**: dynamics are deterministic; use `env.adjacent_dangerous`.

</details>

<details>
<summary>Exercise 3 ★★ — Pre-shield filtering</summary>

> **Hint**: iterate through preferences; return first one whose lookahead is safe.

</details>

<details>
<summary>Exercise 4 ★★ — Shielded vs unshielded Q-learning</summary>

No inline hint is supplied for this exercise. Use the corresponding derivation and code bridge in the teaching guide.

</details>

<details>
<summary>Exercise 5 ★★ — Check a finite rollout against the stated safety property</summary>

No inline hint is supplied for this exercise. Use the corresponding derivation and code bridge in the teaching guide.

</details>

<details>
<summary>Exercise 6 ★★ — Meaningful objective: DR vs fixed-length pendulum</summary>

> **Hint**: this is the matching/safety hook — domain randomisation changes the training distribution. Compare its average and worst measured evaluation return; success under a new shift is an empirical question.

</details>

<details>
<summary>Exercise 7 ★★★ — CBF projection in 1D closed form</summary>

> **Hint**: only project when the constraint is violated; otherwise pass through.

</details>

<details>
<summary>Exercise 8 ★★★ — Adversarial OOD evaluation</summary>

> **Hint**: plot return against evaluation length and mark the training interval. There is no universal distance or multiplier beyond which a policy must fail; report this model’s observed failure cases.

</details>

After solving, change one input or assumption and predict the result before
running it. In reference mode, passing checks verifies the demonstration. In learner mode,
the selected checks must pass against your completed functions without fallback.


## Statistical investigation — Zero measured violations still permits nonzero risk

**Goal.** Interpret a zero-event evaluation without equating it with zero population risk. This bounded reference investigation runs on CPU without downloads.
Write your prediction in a new Markdown cell before executing it.

**Predict.** Suppose a frozen controller has zero unsafe episodes in n independent,
identically distributed evaluation episodes. Derive the one-sided 95% upper
bound 1-0.05^(1/n) by solving P(0 events | risk)=0.05. Predict the bound for
n=10,100,1000. Do not count correlated time steps as independent episodes.

**Work a small case by hand.** Let q be the fixed episode-level violation probability. After n
independent Bernoulli episodes, P(0 violations|q)=(1-q)^n. Setting this
probability to 0.05 yields the exact one-sided 95% upper bound
q_U=1-0.05^(1/n). At n=10,100,1000 it is approximately
0.2589,0.02951,0.002991. This confidence construction is not a posterior
95% probability statement about q.

**Reading bridge:** [Probability and statistics lesson](../docs/statistics/09-statistical-claims.md).
These are original course examples informed by the textbook sections in the
[statistics source catalog](../docs/statistics/SOURCES.md); textbook PDFs are not required to run them.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from common.plotting import RL_PALETTE, use_rl_style
use_rl_style()
from common.statistics import bernoulli_zero_event_upper
counts=np.arange(1,1001);bounds=np.array([bernoulli_zero_event_upper(int(n)) for n in counts])
for n in [10,100,1000]:
    upper=bernoulli_zero_event_upper(n)
    assert np.isclose((1-upper)**n,.05)
    print(f'0/{n} independent unsafe episodes: one-sided 95% upper risk={upper:.5f}')
fig,ax=plt.subplots(figsize=(7,3.5))
ax.loglog(counts,bounds,label='Exact one-sided 95% binomial upper bound')
ax.loglog(counts[2:],3/counts[2:],'--',label='Large-n approximation 3/n')
ax.set(xlabel='Independent evaluation episodes with zero unsafe events',ylabel='Episode-risk upper bound')
ax.legend(fontsize=8);fig.tight_layout();plt.show()


**Interpret and transfer.** The bound concerns the stated IID evaluation population. It cannot establish
safety after a model change, under an untested shift, or when episodes share
an unmodeled common cause. A formal shield guarantee instead rests on its
transition-model and feasible-action assumptions.

**Transfer.** Keep the same 100 trajectories but record them ten times. Explain
why the bound remains the n=100 bound. Then state what evidence is needed to
claim a guarantee under stochastic transition support.


**Read your result.** Read the exact curve rather than replacing zero observed events with zero
risk. The 3/n curve is labeled a large-n approximation and is only drawn
where it is at most one. All episode counts rely on the stated IID design.

<details><summary>Worked feedback for the changed assumption</summary>

Recording the same 100 trajectories ten times leaves n=100 and the bound
0.02951. A shield's formal guarantee instead requires safe initialization,
a sound model of every allowed transition, a nonempty feasible action set,
and correct execution. A likely-safe next state is insufficient for a
guarantee over stochastic transition support.

</details>
